# Stage 02 · Step 2 — Supervised RUL fine-tuning

Mount the SSL-pretrained encoder, attach a small regression head, and predict per-component remaining useful life (`rul_C1..rul_C6`) from a 30-day window of telemetry.

Two regimes are trained back-to-back:
1. **`ssl_frozen`** — freeze the encoder; train the head only. This isolates pretext-task transfer.
2. **`scratch`** — randomly initialise the encoder; train end-to-end. This is the no-SSL baseline that Stage 02 must beat.

Validation: sliding-cumulative time-series CV inside printers 70..84 (`ml_models.lib.splits.expanding_window_folds`). Final metrics on test printers 85..99 (held out throughout).

In [ ]:
from __future__ import annotations
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.splits import expanding_window_folds
from ml_models import PROJECT_ROOT
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)
scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg_dict = saved['patch_cfg']
train_cfg = saved['train_cfg']
CONTEXT_LENGTH = int(train_cfg['context_length'])
RUL_COLS = [f'rul_{component_id}' for component_id in COMPONENT_IDS]
feat_fleet[RUL_COLS] = fleet[RUL_COLS].astype('float32').to_numpy()
feat_fleet.head(3)

In [ ]:
RUL_CLIP = 365.0  # cap RUL at 1 year for stable regression

class RULDataset(Dataset):
    def __init__(self, df: pd.DataFrame, printer_ids, day_range: range,
                 feature_cols: list[str], rul_cols: list[str],
                 context_length: int, mean: np.ndarray, std: np.ndarray):
        self.context_length = context_length
        self.feature_cols = feature_cols
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        keep = filter_printers(df, printer_ids)
        keep = keep[(keep['day'] >= day_range.start) & (keep['day'] < day_range.stop)]
        self.samples: list[tuple[np.ndarray, np.ndarray]] = []
        for printer_id, group in keep.groupby('printer_id', sort=False):
            arr = group[feature_cols].to_numpy(dtype=np.float32)
            ruls = group[rul_cols].to_numpy(dtype=np.float32)
            T = arr.shape[0]
            if T < context_length:
                continue
            for end in range(context_length, T):
                window = arr[end - context_length:end]
                target = ruls[end - 1]
                if np.isnan(target).all():
                    continue
                self.samples.append((window, target))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        window, target = self.samples[idx]
        normed = (window - self.mean) / self.std
        clipped = np.minimum(np.where(np.isnan(target), RUL_CLIP, target), RUL_CLIP)
        mask = (~np.isnan(target)).astype(np.float32)
        return (
            torch.from_numpy(normed),
            torch.from_numpy(clipped.astype(np.float32) / RUL_CLIP),
            torch.from_numpy(mask),
        )

In [ ]:
def build_regression_model(load_pretrained: bool) -> PatchTSTForRegression:
    patch_cfg = PatchTSTConfig(**patch_cfg_dict)
    patch_cfg.num_targets = len(RUL_COLS)
    patch_cfg.prediction_length = 1  # we predict 6 RULs per window, not a sequence
    patch_cfg.use_cls_token = False
    model = PatchTSTForRegression(patch_cfg)
    if load_pretrained:
        state = torch.load(MODELS_DIR / 'ssl_encoder.pt', map_location='cpu')
        encoder_state = {k: v for k, v in state.items() if k.startswith('model.')}
        missing, unexpected = model.load_state_dict(encoder_state, strict=False)
        print(f'load_pretrained={load_pretrained}: missing={len(missing)} unexpected={len(unexpected)}')
    return model.to(device)

In [ ]:
@dataclass
class FineTuneCfg:
    epochs: int = 6
    batch_size: int = 128
    lr_head: float = 5e-4
    lr_backbone: float = 5e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

ft = FineTuneCfg()

def train_one_fold(model: nn.Module, train_loader: DataLoader, freeze_encoder: bool) -> nn.Module:
    if freeze_encoder:
        for name, param in model.named_parameters():
            if not name.startswith('regression_head') and 'head' not in name:
                param.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    optim = torch.optim.AdamW(params, lr=ft.lr_head, weight_decay=ft.weight_decay)
    for epoch in range(ft.epochs):
        model.train()
        running = 0.0
        for x, y, m in train_loader:
            x = x.to(device); y = y.to(device); m = m.to(device)
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
                pred = model(past_values=x).regression_outputs.squeeze(-1)
                if pred.shape != y.shape:
                    pred = pred.view(y.shape)
                loss = ((pred - y) ** 2 * m).sum() / m.sum().clamp(min=1.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, ft.grad_clip)
            optim.step()
            running += float(loss.detach().item())
        print(f'  epoch {epoch:02d} | loss {running / max(1, len(train_loader)):.4f}')
    return model

def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    sse = np.zeros(len(RUL_COLS), dtype=np.float64)
    n = np.zeros(len(RUL_COLS), dtype=np.float64)
    abs_err = np.zeros(len(RUL_COLS), dtype=np.float64)
    with torch.no_grad():
        for x, y, m in loader:
            x = x.to(device); y = y.to(device); m = m.to(device)
            pred = model(past_values=x).regression_outputs.squeeze(-1)
            if pred.shape != y.shape:
                pred = pred.view(y.shape)
            err = (pred - y).cpu().numpy() * RUL_CLIP
            mask = m.cpu().numpy()
            sse += (err ** 2 * mask).sum(axis=0)
            abs_err += (np.abs(err) * mask).sum(axis=0)
            n += mask.sum(axis=0)
    rmse = np.sqrt(sse / np.maximum(n, 1.0))
    mae = abs_err / np.maximum(n, 1.0)
    return {
        'rmse_per_component': dict(zip(RUL_COLS, rmse.tolist())),
        'mae_per_component': dict(zip(RUL_COLS, mae.tolist())),
        'rmse_mean': float(rmse.mean()),
        'mae_mean': float(mae.mean()),
    }

In [ ]:
N_DAYS = int(feat_fleet['day'].max() + 1)
FOLDS = expanding_window_folds(N_DAYS, n_folds=4, min_train_days=1800, val_days=400)
print('Folds:', [(len(t), len(v)) for t, v in FOLDS])

cv_results = {'ssl_frozen': [], 'scratch': []}
for variant in ['ssl_frozen', 'scratch']:
    print(f'\n=== variant: {variant} ===')
    for fold_idx, (train_range, val_range) in enumerate(FOLDS):
        print(f'-- fold {fold_idx}: train {train_range.start}..{train_range.stop} val {val_range.start}..{val_range.stop} --')
        train_ds = RULDataset(feat_fleet, TRAIN_PRINTERS, train_range,
                              feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
        val_ds = RULDataset(feat_fleet, VAL_PRINTERS, val_range,
                            feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
        if len(train_ds) == 0 or len(val_ds) == 0:
            print('  empty split, skipping fold')
            continue
        train_loader = DataLoader(train_ds, batch_size=ft.batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=ft.batch_size, shuffle=False)
        model = build_regression_model(load_pretrained=(variant == 'ssl_frozen'))
        train_one_fold(model, train_loader, freeze_encoder=(variant == 'ssl_frozen'))
        metrics = evaluate(model, val_loader)
        metrics['fold'] = fold_idx
        cv_results[variant].append(metrics)
        print('  val', {k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items() if not isinstance(v, dict)})
with (RESULTS_DIR / 'cv_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(cv_results, handle, indent=2)

In [ ]:
test_metrics: dict[str, dict] = {}
for variant in ['ssl_frozen', 'scratch']:
    print(f'\n=== test eval: {variant} ===')
    train_range = range(0, N_DAYS - 365)
    test_range = range(N_DAYS - 365, N_DAYS)
    train_ds = RULDataset(feat_fleet, TRAIN_PRINTERS, train_range,
                          feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
    test_ds = RULDataset(feat_fleet, TEST_PRINTERS, test_range,
                         feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
    train_loader = DataLoader(train_ds, batch_size=ft.batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=ft.batch_size, shuffle=False)
    model = build_regression_model(load_pretrained=(variant == 'ssl_frozen'))
    train_one_fold(model, train_loader, freeze_encoder=(variant == 'ssl_frozen'))
    test_metrics[variant] = evaluate(model, test_loader)
    if variant == 'ssl_frozen':
        torch.save(model.state_dict(), MODELS_DIR / 'rul_head_ssl.pt')
with (RESULTS_DIR / 'test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(test_metrics, handle, indent=2)
test_metrics

In [ ]:
summary = pd.DataFrame(
    {
        variant: {'mae_mean': metrics['mae_mean'], 'rmse_mean': metrics['rmse_mean']}
        for variant, metrics in test_metrics.items()
    }
).T
improvement = (summary.loc['scratch'] - summary.loc['ssl_frozen']) / summary.loc['scratch'] * 100
print(summary.round(3))
print('\nSSL improvement over scratch (% lower error, larger = better):')
print(improvement.round(2))